# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

In [1]:
# The paper (Moro et al.) describes 17 campaigns run between May 2008 and Nov 2010 (data from 2008-2013 in the full dataset)
# Verify via the months/rows present in the data after loading in Problem 2.

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [2]:
import pandas as pd
import numpy as np
import time
df = pd.read_csv('data/bank-additional-full.csv', sep=';')
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [3]:
df.info()
print(df.isna().sum().sum(), 'NaN values')
print({c:(df[c]=='unknown').sum() for c in df.select_dtypes('object') if (df[c]=='unknown').any()})
print((df.pdays==999).mean(), 'share of pdays==999')

<class 'pandas.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  str    
 2   marital         41188 non-null  str    
 3   education       41188 non-null  str    
 4   default         41188 non-null  str    
 5   housing         41188 non-null  str    
 6   loan            41188 non-null  str    
 7   contact         41188 non-null  str    
 8   month           41188 non-null  str    
 9   day_of_week     41188 non-null  str    
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  str    
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null  float64
 1

/var/folders/n5/j374gbyj1n707rqm0mlxds9r0000gn/T/ipykernel_7584/3040382658.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print({c:(df[c]=='unknown').sum() for c in df.select_dtypes('object') if (df[c]=='unknown').any()})


### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [4]:
# Missing values: no NaNs, but 'unknown' is used as a missing marker in job, marital, education, default, housing, loan
# (default has the most). pdays=999 is a sentinel for 'not previously contacted'.
# Dtypes: object columns are categorical -> encode; y -> binary 0/1. Numeric columns are already numeric.


### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [5]:
# Business objective: predict whether a client will subscribe to a term deposit (y) so the bank can target
# the clients most likely to say yes, reducing the number of calls (cost) and increasing conversion.
df.y.value_counts(normalize=True)

y
no     0.887346
yes    0.112654
Name: proportion, dtype: float64

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [6]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
bank_cols = ['age','job','marital','education','default','housing','loan']
X = df[bank_cols]
y = (df.y=='yes').astype(int)
cat = X.select_dtypes('object').columns.tolist()
num = ['age']
prep = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num', StandardScaler(), num)])

/var/folders/n5/j374gbyj1n707rqm0mlxds9r0000gn/T/ipykernel_7584/243202568.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = X.select_dtypes('object').columns.tolist()


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(28831, 7) (12357, 7)


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [8]:
baseline = max(y_test.mean(), 1-y_test.mean())
print(f'Baseline (always predict "no") accuracy: {baseline:.4f}')

Baseline (always predict "no") accuracy: 0.8874


### Problem 9: Score the Model

What is the accuracy of your model?

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
lgr = Pipeline([('prep',prep),('clf',LogisticRegression(max_iter=1000))]).fit(X_train,y_train)

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [10]:
print('Train acc', lgr.score(X_train,y_train)); print('Test acc', lgr.score(X_test,y_test))

Train acc 0.8873434844438278
Test acc 0.8873512988589464


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [11]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
models = {'Logistic Regression':LogisticRegression(max_iter=1000),'KNN':KNeighborsClassifier(),'Decision Tree':DecisionTreeClassifier(random_state=42),'SVM':SVC()}
rows=[]
for n,m in models.items():
    pipe = Pipeline([('prep',prep),('clf',m)])
    t=time.time(); pipe.fit(X_train,y_train); t=time.time()-t
    rows.append([n,t,pipe.score(X_train,y_train),pipe.score(X_test,y_test)])
results = pd.DataFrame(rows,columns=['Model','Train Time','Train Accuracy','Test Accuracy'])
results

,Model,Train Time,Train Accuracy,Test Accuracy
0,Logistic Regression,0.026967,0.887343,0.887351
1,KNN,0.014850,0.890916,0.875698
2,Decision Tree,0.137496,0.918837,0.864692
3,SVM,17.667027,0.888211,0.886785


##### Questions

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score
# Accuracy is misleading with ~89% 'no', so tune on F1 / ROC-AUC. SVM is slow so tune on a subsample.
grids = {
 'Logistic Regression':(LogisticRegression(max_iter=1000,class_weight='balanced'),{'clf__C':[0.01,0.1,1,10]}),
 'KNN':(KNeighborsClassifier(),{'clf__n_neighbors':[5,15,31,51],'clf__weights':['uniform','distance']}),
 'Decision Tree':(DecisionTreeClassifier(random_state=42,class_weight='balanced'),{'clf__max_depth':[3,5,8,12],'clf__min_samples_leaf':[10,50,100]}),
 'SVM':(SVC(class_weight='balanced'),{'clf__C':[0.1,1],'clf__kernel':['rbf','linear']}),
}
rows=[]
for n,(m,g) in grids.items():
    pipe=Pipeline([('prep',prep),('clf',m)])
    Xt,yt=(X_train,y_train) if n!='SVM' else (X_train.sample(6000,random_state=1),y_train.loc[X_train.sample(6000,random_state=1).index])
    gs=GridSearchCV(pipe,g,scoring='roc_auc' if n!='SVM' else 'f1',cv=3,n_jobs=-1)
    t=time.time(); gs.fit(Xt,yt); t=time.time()-t
    pred=gs.predict(X_test)
    rows.append([n,gs.best_params_,t,gs.best_score_,f1_score(y_test,pred),(pred==y_test).mean()])
tuned=pd.DataFrame(rows,columns=['Model','Best Params','Search Time','CV Score','Test F1','Test Accuracy'])
tuned

,Model,Best Params,Search Time,CV Score,Test F1,Test Accuracy
0,Logistic Regression,{'clf__C': 0.01},1.617699,0.647210,0.255411,0.579591
1,KNN,"{'clf__n_neighbors': 51, 'clf__weights': 'unif...",44.039514,0.624187,0.035837,0.886785
2,Decision Tree,"{'clf__max_depth': 8, 'clf__min_samples_leaf':...",0.487579,0.651386,0.264606,0.649591
3,SVM,"{'clf__C': 0.1, 'clf__kernel': 'rbf'}",2.104956,0.258437,0.258576,0.583718


##### Answers
- **P1:** The paper reports 17 marketing campaigns (May 2008 to Nov 2010) in the dataset.
- **P3:** No NaNs, but 'unknown' acts as missing in several categorical columns; pdays=999 is a sentinel. Categoricals need encoding and y needs mapping to 0/1.
- **P7:** Baseline is the majority class ('no'), ~88.7% accuracy.
- **P9-P11:** See outputs above. With only bank-client features, models barely beat the baseline in accuracy; accuracy is a poor metric given class imbalance, so F1/ROC-AUC with class weighting is more informative. `duration` should be excluded for a realistic model.